In [1]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr

df = pd.read_csv('PartA_province_year.csv')   # or the full path from Option 2
print(df.shape)        # expect (70, 6)
df.head()

(70, 7)


,Geography,Year,GDP_real_M,Population,Immigrants,NetNPR,imm_qtrs
0,Newfoundland and Labrador,2018,32836.0,528751.0,1527.0,62.0,NaN
1,Newfoundland and Labrador,2019,34157.0,528101.0,1857.0,588.0,NaN
2,Newfoundland and Labrador,2020,32512.0,527224.0,898.0,-1145.0,NaN
3,Newfoundland and Labrador,2021,32836.0,526872.0,2049.0,1325.0,NaN
4,Newfoundland and Labrador,2022,32488.0,530628.0,3495.0,3919.0,NaN


In [2]:
df = df.sort_values(['Geography','Year'])
g = df.groupby('Geography')

df['pop_change'] = g['Population'].diff()          # people added vs last year
df['gdp_growth'] = g['GDP_real_M'].pct_change()    # % real GDP growth
df['pop_growth'] = g['Population'].pct_change()     # % population growth
df['newcomers']  = df['Immigrants'] + df['NetNPR']  # immigration + net NPR
df['gdp_pp']     = df['GDP_real_M']*1e6 / df['Population']   # real GDP per person

df.head(8)

,Geography,Year,GDP_real_M,Population,Immigrants,NetNPR,imm_qtrs,pop_change,gdp_growth,pop_growth,newcomers,gdp_pp
56,Alberta,2018,338965.0,4293480.0,42014.0,-1561.0,NaN,NaN,NaN,NaN,40453.0,78948.778147
57,Alberta,2019,339281.0,4348515.0,43717.0,6044.0,NaN,55035.0,0.000932,0.012818,49761.0,78022.267372
58,Alberta,2020,312726.0,4404480.0,22970.0,-8547.0,NaN,55965.0,-0.078268,0.012870,14423.0,71001.798169
59,Alberta,2021,329067.0,4431654.0,39391.0,2193.0,NaN,27174.0,0.052253,0.006170,41584.0,74253.766201
60,Alberta,2022,348375.0,4505411.0,49535.0,47718.0,NaN,73757.0,0.058675,0.016643,97253.0,77323.689226
61,Alberta,2023,357012.0,4669816.0,57588.0,93213.0,NaN,164405.0,0.024792,0.036491,150801.0,76450.977940
62,Alberta,2024,367719.0,4881402.0,66394.0,70012.0,NaN,211586.0,0.029991,0.045309,136406.0,75330.611984
63,British Columbia,2018,292182.0,5021609.0,44854.0,28027.0,NaN,NaN,NaN,NaN,72881.0,58184.936342


In [3]:
df['newcomer_share'] = df['newcomers'] / df['pop_change'] * 100

# boom year, sorted
df[df.Year==2023].set_index('Geography')['newcomer_share'].round(0).sort_values(ascending=False)

Geography
Saskatchewan                 133.0
Newfoundland and Labrador    132.0
Ontario                      127.0
Manitoba                     127.0
Quebec                       124.0
British Columbia             119.0
New Brunswick                105.0
Alberta                       92.0
Nova Scotia                   92.0
Prince Edward Island          91.0
Name: newcomer_share, dtype: float64

In [4]:
d = df.dropna(subset=['gdp_growth','pop_growth'])   # drop 2018 (no prior year)

r, p = pearsonr(d['pop_growth'], d['gdp_growth'])
slope = np.polyfit(d['pop_growth'], d['gdp_growth'], 1)[0]
print(f"GDP growth vs population growth:  r = {r:.2f},  p = {p:.3f},  elasticity = {slope:.2f}")

# robustness: does it hold without Ontario?
d2 = d[d.Geography != 'Ontario']
r2, _ = pearsonr(d2['pop_growth'], d2['gdp_growth'])
print(f"Excluding Ontario:  r = {r2:.2f}")

# and GDP growth vs immigration intensity
d = d.assign(imm_intensity = d['newcomers'] / (d['Population'] - d['pop_change']))
r3, p3 = pearsonr(d['imm_intensity'], d['gdp_growth'])
print(f"GDP growth vs immigration intensity:  r = {r3:.2f},  p = {p3:.3f}")

GDP growth vs population growth:  r = 0.20,  p = 0.120,  elasticity = 0.69
Excluding Ontario:  r = 0.23
GDP growth vs immigration intensity:  r = 0.53,  p = 0.000


In [5]:
first = df[df.Year==2018].set_index('Geography')
last  = df[df.Year==2024].set_index('Geography')

dec = pd.DataFrame({
    'gdp_total_%':       np.log(last.GDP_real_M / first.GDP_real_M) * 100,
    'population_part_%': np.log(last.Population  / first.Population) * 100,
    'per_person_part_%': np.log(last.gdp_pp      / first.gdp_pp)     * 100,
}).round(1)

print("Provinces with negative per-person GDP growth:", (dec['per_person_part_%'] < 0).sum(), "of 10")
dec.sort_values('gdp_total_%', ascending=False)

Provinces with negative per-person GDP growth: 4 of 10


,gdp_total_%,population_part_%,per_person_part_%
Geography,,,
Prince Edward Island,20.6,16.0,4.6
British Columbia,15.5,11.8,3.6
Nova Scotia,15.3,11.4,4.0
Quebec,11.8,6.7,5.0
Ontario,11.6,11.6,0.0
New Brunswick,9.6,10.3,-0.7
Alberta,8.1,12.8,-4.7
Saskatchewan,8.1,7.3,0.9
Manitoba,7.5,9.5,-2.0
